In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import joblib
import mlflow
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.isotonic import IsotonicRegression
from scipy.stats import rankdata

from v2_eval import iter_block_splits, within_job_spearman

feat = pd.read_parquet("../data/processed/v2_features.parquet")
BEST_THREE = ["semantic_similarity", "education_similarity", "skills_soft_coverage"]

feat["rank_within_job"] = feat.groupby("job_id")["matched_score"].transform(lambda s: rankdata(s) / len(s))

In [2]:
def evaluate_alpha(feat, alpha):
    rows = []
    for rf, jf, tr, te in iter_block_splits(feat["resume_fold"], feat["job_fold"]):
        est = Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", Ridge(alpha=alpha, random_state=42)),
        ])
        est.fit(feat.loc[tr, BEST_THREE], feat.loc[tr, "rank_within_job"])
        y_pred = est.predict(feat.loc[te, BEST_THREE])
        rows.append(within_job_spearman(feat.loc[te, "matched_score"], y_pred, feat.loc[te, "job_id"]))
    return pd.Series(rows)

alpha_results = {}
for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    res = evaluate_alpha(feat, alpha)
    alpha_results[alpha] = res
    print(f"alpha={alpha:<8} spearman={res.mean():.3f}  std={res.std():.3f}")

best_alpha = max(alpha_results, key=lambda a: alpha_results[a].mean())
print(f"\nMeilleur alpha : {best_alpha}")

alpha=0.001    spearman=0.366  std=0.093
alpha=0.01     spearman=0.366  std=0.093
alpha=0.1      spearman=0.366  std=0.093
alpha=1.0      spearman=0.366  std=0.093
alpha=10.0     spearman=0.367  std=0.093
alpha=100.0    spearman=0.368  std=0.092

Meilleur alpha : 100.0


In [3]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("resume-ranking")
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="ridge_v2_rank_tuned"):
    mlflow.log_param("features", BEST_THREE)
    mlflow.log_param("target", "rank_within_job")
    mlflow.log_param("alpha", best_alpha)

    for alpha, res in alpha_results.items():
        mlflow.log_metric(f"cv_spearman_alpha_{alpha}", res.mean())

    mlflow.log_metric("cv_spearman_mean", alpha_results[best_alpha].mean())
    mlflow.log_metric("cv_spearman_std", alpha_results[best_alpha].std())

    mlflow.set_tag("architecture", "v2 - sentence-transformers, pas de job_position_name")
    mlflow.set_tag(
        "known_limitation",
        "With only 3 informative features and ~6,000 training rows per split, "
        "none of the tested models (Ridge/RF/GB) significantly outperforms a simple average "
        "of the 3 features (Spearman ~0.37). Ridge on rank is retained because it matches the average "
        "without hand-picked weights. Validated on batuhanmtl/job_resume_fit: "
        "correlation 0.55-0.71 with 3 independent scoring methods (vs ~0.05 for v1)."
    )
    mlflow.set_tag("external_validation_dataset", "batuhanmtl/job_resume_fit")
    mlflow.set_tag("external_validation_ai_match_score_rho", "0.706")

    run_id = mlflow.active_run().info.run_id
    print(f"Run MLflow : {run_id}")

Run MLflow : d062f6d762ca486685fa3dbf984c4eac
🏃 View run ridge_v2_rank_tuned at: http://localhost:5001/#/experiments/1/runs/d062f6d762ca486685fa3dbf984c4eac
🧪 View experiment at: http://localhost:5001/#/experiments/1


In [4]:
final_estimator = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", Ridge(alpha=best_alpha, random_state=42)),
])
final_estimator.fit(feat[BEST_THREE], feat["rank_within_job"])

raw_rank_pred = final_estimator.predict(feat[BEST_THREE])

# Calibration: maps the output (a relative rank) to the 0–1 matched_score scale,
# while preserving the order (which matters for ranking) while producing values
# whose distribution resembles the original score.
calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
calibrator.fit(raw_rank_pred, feat["matched_score"])

calibrated_score = calibrator.predict(raw_rank_pred)
print(pd.Series(calibrated_score).describe())
print(f"\nOriginal vs calibrated Spearman (should be ~1.0; calibration does not change the order) : "
      f"{pd.Series(raw_rank_pred).corr(pd.Series(calibrated_score), method='spearman'):.4f}")

2026/10/02 18:43:18 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2528699293b54054bc6bc95b485426a1', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/10/02 18:43:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/02 18:43:32 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\erb\AppData\Local\Temp\tmpdwy4w0y4\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/1

🏃 View run debonair-loon-838 at: http://localhost:5001/#/experiments/1/runs/2528699293b54054bc6bc95b485426a1
🧪 View experiment at: http://localhost:5001/#/experiments/1


2026/10/02 18:43:36 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8a7725ac6b814d4bbc6ca6795704fcb7', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/10/02 18:43:36 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2026/10/02 18:43:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/02 18:43:46 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\erb\AppData\Local\Temp\tmp0_t1vhsm\model

🏃 View run polite-bug-812 at: http://localhost:5001/#/experiments/1/runs/8a7725ac6b814d4bbc6ca6795704fcb7
🧪 View experiment at: http://localhost:5001/#/experiments/1
count    9544.000000
mean        0.660831
std         0.050885
min         0.591378
25%         0.602847
50%         0.659289
75%         0.702519
max         0.813718
dtype: float64

Original vs calibrated Spearman (should be ~1.0; calibration does not change the order) : 0.9966


In [5]:
joblib.dump(final_estimator, "../data/processed/v2_ridge_estimator.pkl")
joblib.dump(calibrator, "../data/processed/v2_score_calibrator.pkl")
print("Model and calibrator saved.")

Model and calibrator saved.
